# 🐼 02 - Plotting with Pandas

Most charts in an analysis start from a table you've already built: a `value_counts()`, a `groupby()`, a `crosstab()`, a `pivot_table()`. Pandas can plot any of them in **one line** with `.plot`.

There's no new library here. `.plot` calls Matplotlib for you and hands back the same `ax` object from notebook 01, so everything you learned there (titles, formatters, `bar_label`, saving) still works.

## ✅ Learning Goals
- Plot a Series or DataFrame with `.plot.<kind>()`, and customize the returned `ax`
- Turn Module 07 summaries (`value_counts`, `groupby`, `crosstab`, `pivot_table`) straight into charts
- Explain how pandas uses the **index** and **columns** when plotting
- Build grouped and 100%-stacked bar charts from a crosstab
- Place pandas plots into a multi-panel figure with `ax=`
- Tell **wide** data from **long** data, and reshape with `.melt()` (the bridge to Seaborn)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from pathlib import Path

OUT = Path('02-OutputFiles')
OUT.mkdir(exist_ok=True)

titanic = pd.read_csv('titanic.csv')
titanic['class'] = titanic['Pclass'].map({1: '1st', 2: '2nd', 3: '3rd'})
poke = pd.read_csv('pokemon.csv')

## 1. The Pattern: Summarize, Then `.plot`
`.plot.<kind>()` works on any Series or DataFrame:

| Method | Chart |
|---|---|
| `.plot.line()` | line (the default for plain `.plot()`) |
| `.plot.bar()` / `.plot.barh()` | vertical / horizontal bars |
| `.plot.hist()` | histogram |
| `.plot.box()` | box plot |
| `.plot.scatter(x=, y=)` | scatter (DataFrame only) |
| `.plot.pie()` | pie (use sparingly, Foundations 07) |

It returns an `ax`. Keep it, and finish the chart with the notebook 01 methods:

In [ ]:
ax = titanic['Age'].plot.hist(bins=20, edgecolor='white', figsize=(8, 4))
ax.set(title='Ages of Titanic Passengers', xlabel='Age (years)', ylabel='Passengers')
plt.show()

Unlike `ax.hist()` in notebook 01, pandas drops the `NaN`s for you. Convenient, but remember they're gone: this chart shows 714 passengers, not 891.

## 2. Bar Charts from `value_counts()`
For bars, pandas uses the Series **index** as the category labels and the **values** as the bar lengths:

In [ ]:
type_counts = poke['Type 1'].value_counts()

ax = type_counts.sort_values().plot.barh(figsize=(7, 6), color='tab:blue')
ax.set(title='Pokémon by Primary Type', xlabel='Number of Pokémon', ylabel='')
ax.spines[['top', 'right']].set_visible(False)
plt.show()

Two habits:
- **Horizontal bars for long or many labels.** 18 type names would overlap on a vertical axis.
- **Sort the bars.** `.sort_values()` before `.barh()` puts the biggest at the top, so ranking is instant. (A nominal category has no natural order, so order by value. An **ordinal** category keeps its own order, as in the next cell.)

In [ ]:
ax = poke['Generation'].value_counts().sort_index().plot.bar(color='tab:blue', rot=0, figsize=(6, 4))
ax.bar_label(ax.containers[0])
ax.set(title='Pokémon per Generation', xlabel='Generation', ylabel='Pokémon')
plt.show()

`.sort_index()` keeps generations 1–6 in order. `rot=0` keeps the tick labels horizontal, and `ax.containers[0]` is the set of bars pandas drew, which is what `bar_label()` needs.

## 3. Bar Charts from `groupby()`
Any one-column group summary plots the same way. Module 07 notebook 03 said to compare skewed groups with the **median**:

In [ ]:
median_attack = poke.groupby('Type 1')['Attack'].median().sort_values()

ax = median_attack.plot.barh(figsize=(7, 6), color='tab:blue')
ax.axvline(poke['Attack'].median(), color='black', linestyle='--', label='all Pokémon')
ax.set(title='Median Attack by Primary Type', xlabel='Median Attack', ylabel='')
ax.legend()
plt.show()

✅ **Your Turn:** Module 07 notebook 04 warned about small groups. Redo this chart for **types with at least 30 Pokémon only**, and add each type's count to its label (hint: build labels like `f'{name} (n={count})'`).

In [ ]:
# Your turn here

## 4. DataFrames: One Series per Column
When you plot a **DataFrame**, pandas draws **one series per column**, using the **index** for the x-axis. A `pivot_table` from Module 07 is already in this shape: classes down the index, sexes across the columns.

In [ ]:
survival = titanic.pivot_table(values='Survived', index='class', columns='Sex', aggfunc='mean')
survival

In [ ]:
ax = survival.plot.bar(rot=0, figsize=(7, 4), color=['tab:orange', 'tab:blue'])
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.set(title='Survival Rate by Class and Sex', xlabel='Ticket class', ylabel='Survived', ylim=(0, 1))
for bars in ax.containers:                       # one container per column
    ax.bar_label(bars, labels=[f'{v:.0%}' for v in bars.datavalues], padding=2)
plt.show()

That's the Module 07 pivot table in one chart. The pattern (women survived far more in every class, and 3rd class did worst for both) is now visible at a glance.

To group by sex instead, **transpose** the table with `.T`, so the sexes become the index:

In [ ]:
ax = survival.T.plot.bar(rot=0, figsize=(7, 4))
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.set(title='Survival Rate by Sex and Class', xlabel='', ylabel='Survived', ylim=(0, 1))
ax.legend(title='Class')
plt.show()

Same numbers, different emphasis. The first chart invites "compare the sexes within each class." The second invites "compare the classes within each sex." **Put the comparison you care about most side by side.**

## 5. 100% Stacked Bars from a Crosstab
A row-normalized crosstab (Module 07: `normalize='index'`) has rows that each add up to 1. Stacked bars of those rows show **composition**:

In [ ]:
composition = pd.crosstab(titanic['Embarked'], titanic['class'], normalize='index')

ax = composition.plot.barh(stacked=True, figsize=(8, 3.5), colormap='viridis')
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.set(title='Ticket Class Mix at Each Port', xlabel='Share of passengers', ylabel='Port')
ax.legend(title='Class', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.show()

This is the confounder from Module 07 notebook 04, in one picture. Cherbourg (`C`) had a far larger share of 1st-class passengers than the other ports, and Queenstown (`Q`) was almost entirely 3rd class.

`bbox_to_anchor=(1.01, 1), loc='upper left'` moves the legend just **outside** the right edge, so it doesn't cover any bars.

## 6. Lines for Ordered x-Values
Lines imply order and connection (Foundations 07). Pokémon generations are ordered, so a line of the mean stats per generation is fair:

In [ ]:
stats = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']
by_generation = poke.groupby('Generation')[stats].mean()

ax = by_generation.plot(marker='o', figsize=(8, 4.5))
ax.set(title='Mean Stats by Generation', xlabel='Generation', ylabel='Mean value')
ax.legend(ncols=6, frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.15))
plt.show()

Six columns, six lines, no loop. The y-axis doesn't start at zero, which is fine for **lines**, since they encode change by position and slope, not length. Just read the axis before you decide a change is "big."

## 7. Scatter Plots
`df.plot.scatter()` needs `x=` and `y=`. A third numeric column can set the color with `c=`:

In [ ]:
ax = poke.plot.scatter(x='Attack', y='Defense', c='Speed', colormap='viridis',
                       alpha=0.7, figsize=(8, 5))
ax.set(title='Attack vs. Defense, Colored by Speed')
plt.show()

## 8. Pandas Plots in a Multi-Panel Figure
Pass `ax=` to draw a pandas plot into a specific panel:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout='constrained')

titanic['class'].value_counts().sort_index().plot.bar(ax=axes[0], rot=0, color='tab:blue')
axes[0].set(title='Passengers per Class', xlabel='')

titanic.groupby('class')['Survived'].mean().plot.bar(ax=axes[1], rot=0, color='tab:green')
axes[1].yaxis.set_major_formatter(PercentFormatter(xmax=1))
axes[1].set(title='Survival Rate per Class', xlabel='', ylim=(0, 1))

titanic.groupby('class')['Fare'].median().plot.bar(ax=axes[2], rot=0, color='tab:purple')
axes[2].set(title='Median Fare per Class (£)', xlabel='')

fig.suptitle('Titanic by Ticket Class')
fig.savefig(OUT / 'titanic_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Wide vs. Long Data: The Bridge to Seaborn
Pandas plotting wants **wide** data: one column per series, with the x-values in the index, like `survival` above.

Seaborn (the next two notebooks) wants **long** (also called "tidy") data: **one row per observation**, with columns that **name** things (`class`, `Sex`) rather than columns that **are** things (`female`, `male`).

In [ ]:
print('WIDE (one column per sex):')
print(survival.round(3))

In [ ]:
long = survival.reset_index().melt(id_vars='class', var_name='Sex', value_name='survival_rate')
print('LONG (one row per class-sex combination):')
print(long.round(3))

`.melt()` unpivots: the `female`/`male` column **names** become values in a new `Sex` column. (`pivot_table` goes the other way: long to wide.)

Good news: the raw `titanic` DataFrame is **already long**, with one row per passenger. That's why Seaborn can often plot raw data directly, with no `groupby` first. Notebook 03 starts there.

| | Pandas `.plot` | Seaborn |
|---|---|---|
| Wants | wide data (columns = series) | long data (columns = variables) |
| Best for | quick charts of a summary table you already built | statistical charts straight from raw rows, with grouping by `hue`, `col`, ... |
| Returns | a Matplotlib `ax` | a Matplotlib `ax` (or a grid of them) |

---
### 📝 Summary
- `.plot.<kind>()` on a Series or DataFrame draws with Matplotlib and returns the `ax`.
- The **index** becomes the x-axis (or bar labels), and each **column** becomes one series.
- Sort bars by value for nominal categories, and keep the natural order for ordinal ones.
- `pivot_table` → grouped bars, and a row-normalized `crosstab` → 100% stacked bars.
- Use `ax=` to place pandas plots into panels.
- Pandas plots **wide** data. Seaborn plots **long** data. `.melt()` converts wide to long.

## 🏋️ Practice

### Practice 1 — Embarkation Ports (easy)
Make a sorted horizontal bar chart of the number of passengers from each `Embarked` port, with bar labels and readable port names (map `C`/`Q`/`S` to `Cherbourg`/`Queenstown`/`Southampton` first).

In [ ]:
# Your code here

### Practice 2 — Legendary Share by Generation (medium)
Use `pd.crosstab(poke['Generation'], poke['Legendary'], normalize='index')` to make a 100% stacked **vertical** bar chart. Format the y-axis as percentages and move the legend outside the plot.

In [ ]:
# Your code here

### Practice 3 — Wide to Long and Back (harder)
1. Build `by_generation` (section 6) and `.melt()` it into long form with columns `Generation`, `stat`, and `mean_value`.
2. Use `.pivot_table()` on your long table to get the original wide table back.
3. Check that they match with `.equals()` (you may need `.round()` and to reorder the columns).

In [ ]:
# Your code here

## ✏️ Your Turn
Build a 2 × 2 figure about **Pokémon types** using only pandas plots with `ax=`: count per `Type 1` (top 10), median `Total` per type (top 10), Legendary share per type (top 10), and a scatter of `Attack` vs. `Sp. Atk`. Give every panel a title, and save it as `OUT / 'pokemon_types.png'`.

In [ ]:
# Your turn here

## 🔥 Challenge (Optional) — Error Bars from a Summary Table
`.plot.bar()` accepts `yerr=` for error bars. Compute the mean **and** std of `Total` per `Generation` with `.agg(['mean', 'std'])`, then plot the means with `yerr=` set to the std column. What does the error bar show that the bar alone hides? (Module 07 notebook 02: center **and** spread.)

In [ ]:
# Your code here

## 📦 Check Your Output Files
Run this last. Everything you saved should be inside `02-OutputFiles`. Upload that folder along with this notebook.

In [ ]:
for path in sorted(OUT.rglob('*')):
    print(path)

---
✅ Next up → [Seaborn: Distributions](03-Seaborn_Distributions.ipynb): plotting the **shape** of the data straight from raw rows, and comparing it across groups.